# 05 · Mine hard negatives for fine-tuning

For each synthetic (query, movie) pair from notebook 04:

1. retrieve the top-5 movies for the query with the **base** mxbai index;
2. ask Llama-3.1-8B (Groq) to rank the five titles by relevance to the query;
3. keep the three lowest-ranked titles that are not the correct movie as *hard negatives* (similar in embedding space, judged less relevant by the LLM) and attach their plots.

**Output:** `data/training/training_pairs_hard_negatives.csv` with columns `query, positive, negative_1, negative_2, negative_3` (2,010 rows). Resumable; requires `GROQ_API_KEY`.

Note: the final LoRA runs (notebook 06) train on the (query, positive) columns with in-batch negatives for speed on CPU; the explicit negatives are available for a triplet-loss run on a GPU.

In [ ]:
import numpy as np
import pickle
import faiss
from sentence_transformers import SentenceTransformer
from pathlib import Path
import pandas as pd

In [ ]:
df = pd.read_csv("../data/training/training_step_1.csv")
df2 = pd.read_csv("../data/cleaned_movie_plots_v2.csv")

In [ ]:
EMBEDDINGS_PATH = "../artifacts/embeddings_mxbai_base.npy"
METADATA_PATH = "../artifacts/metadata.pkl"

embeddings = np.load(EMBEDDINGS_PATH).astype("float32")
with open(METADATA_PATH, "rb") as f:
    metadata = pickle.load(f)

# Normalize embeddings so inner product search behaves like cosine similarity.
faiss.normalize_L2(embeddings)

In [ ]:
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)
index.add(embeddings)

In [ ]:
model = SentenceTransformer("mixedbread-ai/mxbai-embed-large-v1")

In [ ]:
def embed_query(text):
    prefixed = f"Represent this sentence for searching relevant passages: {text}"
    embedding = model.encode([prefixed]).astype("float32")
    print("Query embedding shape:", embedding.shape)
    faiss.normalize_L2(embedding)
    return embedding

In [ ]:
def search_similar_movies(query_text, top_k=5):
    query_vec = embed_query(query_text)
    scores, indices = index.search(query_vec, top_k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        movie = metadata[idx]
        results.append({
            "title": movie["title"],
            # add more metadata fields if you want
        })
    return results

In [ ]:
import os
from groq import Groq
client = Groq(api_key=os.environ["GROQ_API_KEY"])

In [ ]:
import pandas as pd
import json
import re
import time
import os

OUTPUT_CSV = "../data/training/training_pairs_hard_negatives.csv"

# initialize output file
if not os.path.exists(OUTPUT_CSV):
    pd.DataFrame(columns=["query", "positive", "negative_1", "negative_2", "negative_3"]).to_csv(OUTPUT_CSV, index=False)
    print("Starting fresh.")
else:
    print(f"Resuming — {len(pd.read_csv(OUTPUT_CSV))} rows already done.")

# get already processed queries
done_queries = set(pd.read_csv(OUTPUT_CSV)["query"].tolist()) if os.path.exists(OUTPUT_CSV) else set()

# build plot lookup from df2
plot_lookup = df2.set_index("title")["plot"].to_dict()

RANK_PROMPT = """You are a movie expert. Given a search query and a list of 5 movie titles, rank them by relevance to the query.

Query: {query}

Movies:
{titles}

You MUST return ONLY a valid JSON array of exactly 5 title strings, ranked from most relevant to least relevant.
The array must use double quotes only. No explanation, no extra text, just the array.
Example format: ["Title One", "Title Two", "Title Three", "Title Four", "Title Five"]"""
results = []
failed = []

for _, row in df.iterrows():
    query      = str(row["query"]).strip()
    title      = str(row["title"]).strip()
    plot       = str(row["plot"]).strip()

    # skip already done
    if query in done_queries:
        continue

    try:
        # step 1 — get top 5 candidates from FAISS
        candidates = search_similar_movies(query, top_k=5)
        candidate_titles = [c["title"] for c in candidates]

        # make sure candidate titles don't include the correct movie
        # if they do that's fine — Groq will rank it high and we skip it for negatives

        # step 2 — ask Groq to rank them by relevance to the query
        titles_str = "\n".join([f"{i+1}. {t}" for i, t in enumerate(candidate_titles)])
        prompt = RANK_PROMPT.format(query=query, titles=titles_str)

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            response_format={"type": "json_object"},
        )

        text = response.choices[0].message.content.strip()
text = re.sub(r"```json|```", "", text).strip()

# try to extract array even if wrapped in object
array_match = re.search(r'\[.*?\]', text, re.DOTALL)
if array_match:
    ranked = json.loads(array_match.group())
else:
    ranked = json.loads(text)

# clean up any escaped quotes in titles
ranked = [t.replace("\\'", "'") for t in ranked]

        # step 3 — grab last 3 as hard negatives (ranked 3rd, 4th, 5th)
        # filter out the correct movie just in case it appeared in candidates
        negatives = [t for t in ranked if t.lower() != title.lower()][-3:]

        if len(negatives) < 3:
            print(f"✗ Not enough negatives for '{title}' — skipping")
            failed.append(title)
            continue

        # step 4 — fetch plots for negatives from df2
        neg_plots = []
        for neg_title in negatives:
            neg_plot = plot_lookup.get(neg_title, None)
            if neg_plot is None:
                # try fuzzy match
                matches = df2[df2["title"].str.contains(neg_title[:15], case=False, na=False)]
                neg_plot = matches.iloc[0]["plot"] if len(matches) > 0 else "plot not found"
            neg_plots.append(str(neg_plot).strip())

        # step 5 — save row immediately
        new_row = pd.DataFrame([{
            "query":      query,
            "positive":   plot,
            "negative_1": neg_plots[0],
            "negative_2": neg_plots[1],
            "negative_3": neg_plots[2],
        }])
        new_row.to_csv(OUTPUT_CSV, mode="a", header=False, index=False)
        done_queries.add(query)

        print(f"✓ [{len(done_queries)}] '{title}' → negatives: {negatives}")

    except Exception as e:
        print(f"✗ '{title}' — {e}")
        failed.append(title)

    time.sleep(1.5)

print(f"\nDone. {len(done_queries)} rows saved to {OUTPUT_CSV}")
print(f"Failed: {len(failed)}")

In [ ]:
search_similar_movies("A story about a young wizard who discovers his magical heritage and attends a school of witchcraft and wizardry.", top_k=5)